# 04 — Monte Carlo Methods for RL

## Learning Objectives
1. Implement first-visit and every-visit MC prediction from episode samples
2. Build MC control with epsilon-greedy policy improvement
3. Apply off-policy MC using importance sampling for safe policy evaluation
4. Compare MC variants (first/every-visit, on/off-policy) by convergence variance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import List, Tuple, Dict

np.random.seed(42)

try:
    import torch
    torch.manual_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    device = 'cpu'

print(f'numpy {np.__version__}, torch={TORCH_AVAILABLE}')

# --- Simplified BlackJack environment (numpy only, no gym) ---
def draw_card() -> int:
    """Draw a card: value in 1-10 (face cards = 10)."""
    card = np.random.randint(1, 14)  # 1..13
    return min(card, 10)

def blackjack_step(player_sum: int, dealer_card: int, action: int) -> Tuple[int, float, bool]:
    """Simplified BlackJack step.

    Args:
        player_sum: Current player hand sum.
        dealer_card: Visible dealer card (1-10).
        action: 0=stick, 1=hit.

    Returns:
        next_player_sum: Updated player sum after hit.
        reward: 1=win, -1=lose, 0=draw (only when done).
        done: Whether episode ended.
    """
    if action == 1:  # hit: draw another card
        player_sum += draw_card()
    # Episode ends if player sticks or busts
    done = (action == 0) or (player_sum > 21)
    if done:
        if player_sum > 21:  # bust
            return player_sum, -1.0, True
        # Dealer plays: must hit until sum >= 17
        dealer_sum = dealer_card
        while dealer_sum < 17:
            dealer_sum += draw_card()
        if dealer_sum > 21 or player_sum > dealer_sum:
            reward = 1.0   # player wins
        elif player_sum == dealer_sum:
            reward = 0.0   # draw
        else:
            reward = -1.0  # player loses
        return player_sum, reward, True
    return player_sum, 0.0, False  # hit, no reward yet


def reset_blackjack() -> Tuple[int, int]:
    """Start a new BlackJack episode. Returns (player_sum, dealer_card)."""
    player_sum = draw_card() + draw_card()
    dealer_card = draw_card()
    return player_sum, dealer_card


# Test the environment
ps, dc = reset_blackjack()
print(f'Sample episode: player_sum={ps}, dealer_card={dc}')
ps2, r, done = blackjack_step(ps, dc, 1)  # hit
print(f'  Hit -> new sum={ps2}, reward={r}, done={done}')


## Level 1: First-Visit MC Prediction for Fixed Policy

In [ ]:
# State: (player_sum, dealer_card) -> discretized to bins
# Fixed policy: stick at 20+, otherwise hit

def stick_at_20_policy(player_sum: int, dealer_card: int) -> int:
    """Simple fixed policy: stick (0) if player_sum >= 20, else hit (1)."""
    return 0 if player_sum >= 20 else 1


def generate_episode(policy_fn) -> List[Tuple]:
    """Generate one BlackJack episode following policy_fn.

    Returns:
        List of (player_sum, dealer_card, action, reward) tuples.
    """
    trajectory = []
    player_sum, dealer_card = reset_blackjack()
    for _ in range(100):  # max 100 steps per episode
        action = policy_fn(player_sum, dealer_card)
        next_sum, reward, done = blackjack_step(player_sum, dealer_card, action)
        trajectory.append((player_sum, dealer_card, action, reward))
        player_sum = next_sum
        if done:
            break
    return trajectory


def first_visit_mc_prediction(policy_fn, n_episodes=5000, gamma=1.0):
    """First-visit Monte Carlo prediction for a fixed policy.

    Returns:
        V: Dict mapping (player_sum, dealer_card) -> estimated value.
        visit_counts: Number of first-visit updates per state.
    """
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(n_episodes):
        trajectory = generate_episode(policy_fn)
        # Compute discounted returns backwards
        G = 0.0
        visited = set()  # for first-visit: only count first occurrence
        for step in reversed(trajectory):
            ps, dc, a, r = step
            G = gamma * G + r
            state = (ps, dc)
            if state not in visited:  # first-visit MC
                visited.add(state)
                returns_sum[state] += G
                returns_count[state] += 1

    V = {s: returns_sum[s] / returns_count[s] for s in returns_sum}
    return V, returns_count


print('Running first-visit MC prediction (stick-at-20 policy, 10000 episodes)...')
V_mc, counts_mc = first_visit_mc_prediction(stick_at_20_policy, n_episodes=10000)

# Visualize: V(player_sum, dealer_card) for dealer_card=5 (a common card)
psums = list(range(12, 22))  # player sums 12-21
dcards = list(range(1, 11))  # dealer cards 1-10

V_grid = np.zeros((len(psums), len(dcards)))
for i, ps in enumerate(psums):
    for j, dc in enumerate(dcards):
        V_grid[i, j] = V_mc.get((ps, dc), 0.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
im = ax1.imshow(V_grid, cmap='RdYlGn', origin='upper', aspect='auto',
                extent=[0.5, 10.5, 21.5, 11.5])
ax1.set_xlabel('Dealer Card'); ax1.set_ylabel('Player Sum')
ax1.set_title('MC V(s): Stick-at-20 Policy')
ax1.set_xticks(range(1, 11)); ax1.set_yticks(range(12, 22))
plt.colorbar(im, ax=ax1, label='V(player_sum, dealer_card)')

# V vs player_sum for dealer_card=7
v_dealer7 = [V_mc.get((ps, 7), 0.0) for ps in psums]
ax2.plot(psums, v_dealer7, 'o-', color='steelblue', linewidth=2)
ax2.axhline(0, color='black', linestyle=':', alpha=0.4)
ax2.set_xlabel('Player Sum'); ax2.set_ylabel('V(state)')
ax2.set_title('MC V: Dealer Card = 7')
ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_04_mc_prediction.png', dpi=80, bbox_inches='tight')
plt.show()

print(f'Unique states visited: {len(V_mc)}')
print(f'V(sum=20, dealer=7): {V_mc.get((20, 7), "not seen"):.3f}')
print(f'V(sum=12, dealer=7): {V_mc.get((12, 7), "not seen"):.3f}')


## Level 2: MC Control — Epsilon-Greedy Policy Improvement

In [ ]:
def epsilon_greedy_action(Q_vals: np.ndarray, epsilon: float) -> int:
    """Select action epsilon-greedily given Q values for one state."""
    if np.random.random() < epsilon:
        return np.random.randint(len(Q_vals))   # explore
    return int(np.argmax(Q_vals))               # exploit


def mc_control_epsilon_greedy(n_episodes=20000, gamma=1.0, epsilon=0.1):
    """On-policy MC control with epsilon-greedy exploration.

    Maintains Q(s, a) estimates and improves policy greedily.
    Returns Q table and episode reward history for analysis.
    """
    # Q[state][action] -> running average
    Q_sum = defaultdict(lambda: np.zeros(2))
    Q_count = defaultdict(lambda: np.zeros(2))
    episode_rewards = []

    for ep in range(n_episodes):
        # Generate episode with current epsilon-greedy policy
        trajectory = []
        player_sum, dealer_card = reset_blackjack()
        ep_reward = 0.0

        for _ in range(100):
            state = (player_sum, dealer_card)
            Q_s = Q_sum[state] / np.maximum(Q_count[state], 1)  # safe divide
            action = epsilon_greedy_action(Q_s, epsilon)
            next_sum, reward, done = blackjack_step(player_sum, dealer_card, action)
            trajectory.append((state, action, reward))
            ep_reward += reward
            player_sum = next_sum
            if done:
                break

        episode_rewards.append(ep_reward)

        # First-visit MC update
        G = 0.0
        visited_sa = set()
        for state, action, reward in reversed(trajectory):
            G = gamma * G + reward
            if (state, action) not in visited_sa:
                visited_sa.add((state, action))
                Q_sum[state][action] += G
                Q_count[state][action] += 1.0

    Q = {s: Q_sum[s] / np.maximum(Q_count[s], 1) for s in Q_sum}
    return Q, episode_rewards


print('MC Control: epsilon=0.1 (20000 episodes)...')
Q_01, rewards_01 = mc_control_epsilon_greedy(n_episodes=20000, epsilon=0.1)
print('MC Control: epsilon=0.3 (20000 episodes)...')
Q_03, rewards_03 = mc_control_epsilon_greedy(n_episodes=20000, epsilon=0.3)

# Compare win rates (smoothed over 500-episode windows)
def smooth(arr, w):
    return np.convolve(np.array(arr), np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

win_smooth_01 = smooth([1 if r > 0 else 0 for r in rewards_01], 500)
win_smooth_03 = smooth([1 if r > 0 else 0 for r in rewards_03], 500)
axes[0].plot(win_smooth_01, label='epsilon=0.1', color='steelblue', linewidth=2)
axes[0].plot(win_smooth_03, label='epsilon=0.3', color='darkorange', linewidth=2)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Win Rate (500-ep window)')
axes[0].set_title('MC Control: epsilon-Greedy Win Rate')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Extract optimal policy: stick/hit threshold per state
psums_ctrl = list(range(12, 22))
dcards_ctrl = list(range(1, 11))
policy_grid_01 = np.zeros((len(psums_ctrl), len(dcards_ctrl)))
for i, ps in enumerate(psums_ctrl):
    for j, dc in enumerate(dcards_ctrl):
        state = (ps, dc)
        if state in Q_01:
            policy_grid_01[i, j] = Q_01[state].argmax()  # 0=stick, 1=hit

im2 = axes[1].imshow(policy_grid_01, cmap='RdYlGn_r', origin='upper', aspect='auto',
                     extent=[0.5, 10.5, 21.5, 11.5], vmin=0, vmax=1)
axes[1].set_xlabel('Dealer Card'); axes[1].set_ylabel('Player Sum')
axes[1].set_title('Learned Policy (0=Stick, 1=Hit) epsilon=0.1')
axes[1].set_xticks(range(1, 11)); axes[1].set_yticks(range(12, 22))
plt.colorbar(im2, ax=axes[1], ticks=[0, 1], label='Action')
plt.tight_layout(); plt.savefig('rl_04_mc_control.png', dpi=80, bbox_inches='tight')
plt.show()

final_wr_01 = np.mean([r > 0 for r in rewards_01[-5000:]])
final_wr_03 = np.mean([r > 0 for r in rewards_03[-5000:]])
print(f'Final 5000-ep win rate: epsilon=0.1 -> {final_wr_01:.3f}, epsilon=0.3 -> {final_wr_03:.3f}')
print('Lower epsilon -> less exploration -> higher win rate once policy stabilizes.')


## Real-World Example 1: Off-Policy MC with Importance Sampling

In [ ]:
# Off-policy MC: behavior policy = random, target policy = greedy
# Weighted importance sampling (WIS) for variance reduction vs ordinary IS

def random_policy(player_sum: int, dealer_card: int) -> int:
    """Behavior policy: uniform random (50% stick, 50% hit)."""
    return np.random.randint(2)


def greedy_policy_from_Q(Q_table: dict, player_sum: int, dealer_card: int) -> int:
    """Target policy: greedy with respect to Q table."""
    state = (player_sum, dealer_card)
    if state in Q_table:
        return int(Q_table[state].argmax())
    return 0  # default: stick if never seen


def off_policy_mc_wis(target_Q, n_episodes=10000, gamma=1.0):
    """Weighted importance sampling MC prediction for target policy.

    Uses behavior=random, target=greedy(Q), corrects via IS ratio.
    Returns V estimates for key states over episodes.
    """
    # Track WIS estimates for state (18, 7) as running example
    monitor_state = (18, 7)
    G_num = 0.0   # numerator for WIS
    G_den = 0.0   # denominator for WIS
    v_estimates = []

    for ep in range(n_episodes):
        trajectory = []
        player_sum, dealer_card = reset_blackjack()
        state_appears = False
        for _ in range(100):
            state = (player_sum, dealer_card)
            if state == monitor_state:
                state_appears = True
            action = random_policy(player_sum, dealer_card)
            next_sum, reward, done = blackjack_step(player_sum, dealer_card, action)
            trajectory.append((state, action, reward))
            player_sum = next_sum
            if done:
                break

        if not state_appears:
            v_estimates.append(G_num / max(G_den, 1e-10))
            continue

        # Compute IS ratio from monitor_state to end
        G = 0.0
        W = 1.0  # cumulative IS ratio
        found_monitor = False
        for state, action, reward in reversed(trajectory):
            G = gamma * G + reward
            # IS ratio: pi_target(a|s) / pi_behavior(a|s)
            a_target = greedy_policy_from_Q(target_Q, state[0], state[1])
            pi_target = 1.0 if action == a_target else 0.0
            pi_behavior = 0.5  # uniform random
            W *= pi_target / pi_behavior
            if state == monitor_state:
                found_monitor = True
                break

        if found_monitor:
            G_num += W * G
            G_den += W

        v_estimates.append(G_num / max(G_den, 1e-10))

    return v_estimates


print('Off-policy MC (WIS) for state (18,7)...')
wis_estimates = off_policy_mc_wis(Q_01, n_episodes=10000)
true_approx = V_mc.get((18, 7), 0.0)

fig_wis, ax_wis = plt.subplots(figsize=(10, 4))
ax_wis.plot(wis_estimates, color='steelblue', alpha=0.8, linewidth=1.5, label='WIS estimate')
ax_wis.axhline(true_approx, color='red', linestyle='--', linewidth=2, label=f'On-policy MC: {true_approx:.3f}')
ax_wis.set_xlabel('Episode'); ax_wis.set_ylabel('V(18, dealer=7)')
ax_wis.set_title('Off-Policy MC (Weighted IS) Convergence')
ax_wis.legend(); ax_wis.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_04_off_policy_is.png', dpi=80, bbox_inches='tight')
plt.show()

print(f'On-policy MC V(18,7):  {true_approx:.4f}')
print(f'WIS final estimate:    {wis_estimates[-1]:.4f}')
print('WIS reduces variance by normalizing by sum of IS weights (vs ordinary IS).')


## Real-World Example 2: MC Convergence — First-Visit vs Every-Visit Variance

In [ ]:
def every_visit_mc_prediction(policy_fn, n_episodes=5000, gamma=1.0):
    """Every-visit MC prediction: update V at every visit to a state, not just first.

    Lower bias than first-visit in early training, but higher variance.
    """
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(n_episodes):
        trajectory = generate_episode(policy_fn)
        G = 0.0
        for step in reversed(trajectory):
            ps, dc, a, r = step
            G = gamma * G + r
            state = (ps, dc)
            # Every-visit: update without checking first occurrence
            returns_sum[state] += G
            returns_count[state] += 1

    V = {s: returns_sum[s] / returns_count[s] for s in returns_sum}
    return V, returns_count


def run_mc_trials(policy_fn, n_trials=10, n_episodes=5000, mode='first_visit'):
    """Run multiple MC trials to estimate variance of V estimates.

    Returns:
        v_all: Array of shape (n_trials,) — V estimates for a fixed state.
    """
    monitor_state = (18, 7)
    v_all = []
    for trial in range(n_trials):
        np.random.seed(trial * 100)  # different seed per trial
        if mode == 'first_visit':
            V_t, _ = first_visit_mc_prediction(policy_fn, n_episodes=n_episodes)
        else:
            V_t, _ = every_visit_mc_prediction(policy_fn, n_episodes=n_episodes)
        v_all.append(V_t.get(monitor_state, 0.0))
    return np.array(v_all)


# Compare first-visit vs every-visit variance across episode counts
ep_counts = [500, 1000, 2000, 5000, 10000]
fv_means, fv_stds = [], []
ev_means, ev_stds = [], []

print('Comparing first-visit vs every-visit MC variance (5 trials each)...')
for n_ep in ep_counts:
    fv = run_mc_trials(stick_at_20_policy, n_trials=5, n_episodes=n_ep, mode='first_visit')
    ev = run_mc_trials(stick_at_20_policy, n_trials=5, n_episodes=n_ep, mode='every_visit')
    fv_means.append(fv.mean()); fv_stds.append(fv.std())
    ev_means.append(ev.mean()); ev_stds.append(ev.std())
    print(f'  n_ep={n_ep:6d}: FV mean={fv.mean():.3f}+/-{fv.std():.3f}  '
          f'EV mean={ev.mean():.3f}+/-{ev.std():.3f}')

fig_var, axes_var = plt.subplots(1, 2, figsize=(13, 5))

fv_means = np.array(fv_means); fv_stds = np.array(fv_stds)
ev_means = np.array(ev_means); ev_stds = np.array(ev_stds)

axes_var[0].errorbar(ep_counts, fv_means, yerr=fv_stds, label='First-Visit', fmt='o-',
                     color='steelblue', linewidth=2, capsize=5)
axes_var[0].errorbar(ep_counts, ev_means, yerr=ev_stds, label='Every-Visit', fmt='s--',
                     color='darkorange', linewidth=2, capsize=5)
axes_var[0].set_xlabel('Number of Episodes'); axes_var[0].set_ylabel('V estimate (18, dealer=7)')
axes_var[0].set_title('First-Visit vs Every-Visit MC: Mean +/- Std')
axes_var[0].legend(); axes_var[0].grid(True, alpha=0.3)

axes_var[1].plot(ep_counts, fv_stds, 'o-', color='steelblue', label='First-Visit Std', linewidth=2)
axes_var[1].plot(ep_counts, ev_stds, 's--', color='darkorange', label='Every-Visit Std', linewidth=2)
axes_var[1].set_xlabel('Number of Episodes'); axes_var[1].set_ylabel('Std of V estimate')
axes_var[1].set_title('Variance Reduction with More Episodes')
axes_var[1].legend(); axes_var[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_04_mc_variance.png', dpi=80, bbox_inches='tight')
plt.show()


## Real-World Example 3: MC vs TD Comparison on a Simple MDP

In [ ]:
# 5-state random walk: states 0-6, left/right actions, reward=1 at s=6
# True V known: V(s) = s/6 for s in 1-5
# Compare: MC vs TD(0) convergence speed

TRUE_V = np.array([0.0, 1/6, 2/6, 3/6, 4/6, 5/6, 0.0])  # terminal states at 0,6

def random_walk_episode(start=3):
    """Generate random walk episode. Returns list of (state, reward, next_state)."""
    s = start
    trajectory = []
    while True:
        direction = np.random.choice([-1, 1])  # left or right
        ns = s + direction
        reward = 1.0 if ns == 6 else 0.0
        trajectory.append((s, reward, ns))
        s = ns
        if s == 0 or s == 6:  # terminal
            break
    return trajectory


def mc_prediction_rw(n_episodes=500, gamma=1.0):
    """MC prediction on random walk (first-visit). Returns V array shape (7,)."""
    returns_sum = np.zeros(7)
    returns_count = np.zeros(7)
    for _ in range(n_episodes):
        traj = random_walk_episode()
        G = 0.0
        visited = set()
        for s, r, ns in reversed(traj):
            G = gamma * G + r
            if s not in visited:
                visited.add(s)
                returns_sum[s] += G
                returns_count[s] += 1
    V = np.where(returns_count > 0, returns_sum / returns_count, 0.5)
    return V


def td0_prediction_rw(n_episodes=500, alpha=0.1, gamma=1.0):
    """TD(0) prediction on random walk. Returns V array shape (7,)."""
    V = np.full(7, 0.5)  # init to 0.5 (uncertain)
    V[0] = V[6] = 0.0    # terminal states
    for _ in range(n_episodes):
        traj = random_walk_episode()
        for s, r, ns in traj:
            if 0 < s < 6:  # non-terminal
                td_error = r + gamma * V[ns] - V[s]
                V[s] += alpha * td_error
    return V


n_ep_sweep = [10, 50, 100, 200, 500]
mc_rmses, td_rmses = [], []

for n_ep in n_ep_sweep:
    V_mc_rw = mc_prediction_rw(n_episodes=n_ep)
    V_td_rw = td0_prediction_rw(n_episodes=n_ep)
    mc_rmse = np.sqrt(np.mean((V_mc_rw[1:6] - TRUE_V[1:6])**2))
    td_rmse = np.sqrt(np.mean((V_td_rw[1:6] - TRUE_V[1:6])**2))
    mc_rmses.append(mc_rmse); td_rmses.append(td_rmse)
    print(f'  n_ep={n_ep:4d}: MC RMSE={mc_rmse:.4f}, TD RMSE={td_rmse:.4f}')

fig_mc_td, ax_mc_td = plt.subplots(figsize=(9, 4))
ax_mc_td.plot(n_ep_sweep, mc_rmses, 'o-', color='steelblue', label='MC (first-visit)', linewidth=2)
ax_mc_td.plot(n_ep_sweep, td_rmses, 's--', color='darkorange', label='TD(0)', linewidth=2)
ax_mc_td.set_xlabel('Number of Episodes'); ax_mc_td.set_ylabel('RMSE vs True V')
ax_mc_td.set_title('MC vs TD(0): Convergence on Random Walk')
ax_mc_td.legend(); ax_mc_td.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_04_mc_vs_td.png', dpi=80, bbox_inches='tight')
plt.show()

print('\nMC: unbiased but high variance (waits for full episode return).')
print('TD: biased (bootstraps) but lower variance — often faster in practice.')


## Comparison: First-Visit vs Every-Visit MC + Variance Plots

In [ ]:
# Head-to-head comparison of first-visit vs every-visit MC on random walk
# with multiple seeds to show variance distribution

n_trials = 20
n_episodes_cmp = 200
fv_V_all = np.zeros((n_trials, 7))
ev_V_all = np.zeros((n_trials, 7))

for trial in range(n_trials):
    np.random.seed(trial)
    # First-visit
    rs_fv = np.zeros(7); rc_fv = np.zeros(7)
    for _ in range(n_episodes_cmp):
        traj = random_walk_episode()
        G = 0.0; visited = set()
        for s, r, ns in reversed(traj):
            G = G + r
            if s not in visited:
                visited.add(s)
                rs_fv[s] += G; rc_fv[s] += 1
    fv_V_all[trial] = np.where(rc_fv > 0, rs_fv / rc_fv, 0.5)

    # Every-visit
    np.random.seed(trial)
    rs_ev = np.zeros(7); rc_ev = np.zeros(7)
    for _ in range(n_episodes_cmp):
        traj = random_walk_episode()
        G = 0.0
        for s, r, ns in reversed(traj):
            G = G + r
            rs_ev[s] += G; rc_ev[s] += 1
    ev_V_all[trial] = np.where(rc_ev > 0, rs_ev / rc_ev, 0.5)

states_cmp = np.arange(1, 6)
fv_mean = fv_V_all[:, 1:6].mean(axis=0)
fv_std = fv_V_all[:, 1:6].std(axis=0)
ev_mean = ev_V_all[:, 1:6].mean(axis=0)
ev_std = ev_V_all[:, 1:6].std(axis=0)

fig_cmp_mc, axes_mc = plt.subplots(1, 2, figsize=(13, 5))

axes_mc[0].plot(states_cmp, TRUE_V[1:6], 'k--', label='True V', linewidth=2)
axes_mc[0].errorbar(states_cmp - 0.1, fv_mean, yerr=fv_std, label='First-Visit', fmt='o-',
                    color='steelblue', capsize=5, linewidth=2)
axes_mc[0].errorbar(states_cmp + 0.1, ev_mean, yerr=ev_std, label='Every-Visit', fmt='s--',
                    color='darkorange', capsize=5, linewidth=2)
axes_mc[0].set_xlabel('State'); axes_mc[0].set_ylabel('V(s) estimate')
axes_mc[0].set_title(f'FV vs EV MC ({n_trials} trials, {n_episodes_cmp} ep each)')
axes_mc[0].legend(); axes_mc[0].grid(True, alpha=0.3)

# Variance per state
axes_mc[1].bar(states_cmp - 0.15, fv_std, width=0.3, color='steelblue',
               alpha=0.8, label='First-Visit Std')
axes_mc[1].bar(states_cmp + 0.15, ev_std, width=0.3, color='darkorange',
               alpha=0.8, label='Every-Visit Std')
axes_mc[1].set_xlabel('State'); axes_mc[1].set_ylabel('Std of V estimate')
axes_mc[1].set_title('Variance per State: FV vs EV')
axes_mc[1].legend(); axes_mc[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('rl_04_fv_vs_ev.png', dpi=80, bbox_inches='tight')
plt.show()

avg_fv_rmse = np.sqrt(np.mean((fv_mean - TRUE_V[1:6])**2))
avg_ev_rmse = np.sqrt(np.mean((ev_mean - TRUE_V[1:6])**2))
print(f'Average RMSE: FV={avg_fv_rmse:.4f}, EV={avg_ev_rmse:.4f}')
print('First-visit and every-visit converge to same V; FV has lower variance in practice.')


## Key Takeaways

**Core idea:** Monte Carlo methods learn V and Q purely from episode samples — no model needed. They compute exact discounted returns, making them unbiased but high-variance. First-visit MC uses each state once per episode; every-visit uses all occurrences.

| Method | Bias | Variance | Requires | Best for |
|--------|------|----------|---------|----------|
| First-visit MC | None | High | Full episodes | Episodic, moderate state space |
| Every-visit MC | None | Higher | Full episodes | Same; slightly more data usage |
| Off-policy (WIS) | None | Highest | Full episodes + IS | Safe evaluation of target pi |
| TD(0) | Bootstrap | Low | Each step | Continuing tasks, faster |

**Failure modes:**
- High variance with rare episodes: collect more data or use WIS
- Off-policy IS: IS ratio explodes if behavior and target policies diverge (log-IS trick)
- MC not suitable for continuing tasks (no episode boundary)

**Related:** [02-bellman](02-bellman-equations.ipynb), [05-td-learning](05-temporal-difference-learning.ipynb)

## Exercises

1. **Incremental MC:** Replace the full returns_sum/returns_count with an incremental update rule: V(s) <- V(s) + alpha*(G - V(s)). Compare convergence to batch averaging.
2. **Epsilon decay:** In MC control, decay epsilon from 0.5 to 0.01 over training. Does final win rate improve vs fixed epsilon?
3. **Ordinary vs Weighted IS:** Implement ordinary (non-weighted) IS off-policy MC. Compare variance to WIS over 10 trials.
4. **Longer episodes:** The random walk environment terminates at state 0 or 6. Extend to 9 states. How does RMSE compare?